In [1]:
import time
from itertools import chain
import email
import imaplib
import base64
import os
import re

In [4]:
imap_ssl_host = 'imap.gmail.com'
imap_ssl_port = 993
username = 'miguelpinheiro2409@gmail.com'
password = 'TeL3sc0pio'

In [5]:
criteria = {}
uid_max = 0

def search_string(uid_max, criteria):
    c = list(map(lambda t: (t[0], '"'+str(t[1])+'"'), criteria.items())) + [('UID', '%d:*' % (uid_max+1))]
    return '(%s)' % ' '.join(chain(*c))

In [6]:
mail = imaplib.IMAP4_SSL(imap_ssl_host)
mail.login(username, password)
#select the folder
mail.select('inbox')

result, data = mail.uid('SEARCH', None, search_string(uid_max, criteria))
uids = [int(s) for s in data[0].split()]
if uids:
    uid_max = max(uids)
mail.logout()

error: b'[AUTHENTICATIONFAILED] Invalid credentials (Failure)'

In [7]:
while 1:
    mail = imaplib.IMAP4_SSL(imap_ssl_host)
    mail.login(username, password)
    mail.select('inbox')
    result, data = mail.uid('search', None, search_string(uid_max, criteria))
    uids = [int(s) for s in data[0].split()]

    for uid in uids:
        # Have to check again because Gmail sometimes does not obey UID criterion.
        if uid > uid_max:
            result, data = mail.uid('fetch', str(uid), '(RFC822)')
            for response_part in data:
                if isinstance(response_part, tuple):
                    #message_from_string can also be use here
                    print(email.message_from_bytes(response_part[1])) #processing the email here for whatever
            uid_max = uid
mail.logout()
time.sleep(1)

error: b'[AUTHENTICATIONFAILED] Invalid credentials (Failure)'